# Archived Prototype — Wikipedia Mini-1000 + Standard RAG

This notebook contains the earlier **1,000-Wikipedia-article prototype** and its Standard RAG pipeline.

It is kept for reference / sanity testing and is **not the active BeerQA Dev20 experiment**.

> Note: the original notebook code uses `MAX_ARTICLES = 1000` (not 10,000).

## 1. Environment and Paths

Install packages once if they are missing:

`pip install rank-bm25 transformers torch faiss-cpu sentencepiece openai`

In [24]:
from pathlib import Path
import os
import json
import bz2
import re
import getpass
import numpy as np
import torch
import faiss

from rank_bm25 import BM25Okapi
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from openai import OpenAI

WIKI_ROOT = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized"
)

OUTPUT_DIR = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\processed_data"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MINI_CORPUS_PATH = OUTPUT_DIR / "wikipedia_mini_1000_articles.jsonl"

print("Wikipedia folder exists:", WIKI_ROOT.exists())
print("Processed-data folder:", OUTPUT_DIR)

Wikipedia folder exists: True
Processed-data folder: C:\Users\Wei Khang\Documents\Master\FYP\processed_data


## 2. Inspect the Wikipedia `.bz2` Corpus

In [25]:
bz2_files = sorted(WIKI_ROOT.rglob("*.bz2"))

print("Total .bz2 files:", len(bz2_files))
for file in bz2_files[:10]:
    print(file)

if not bz2_files:
    raise FileNotFoundError("No .bz2 Wikipedia files were found under WIKI_ROOT.")

Total .bz2 files: 18013
C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized\AA\wiki_00.bz2
C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized\AA\wiki_01.bz2
C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized\AA\wiki_02.bz2
C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized\AA\wiki_03.bz2
C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized\AA\wiki_04.bz2
C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized\AA\wiki_05.bz2
C:\Users\Wei Khang\Documents\Master\FYP\enwiki-20200801-pages-articles-tokenized\enwiki-20200801-pages-articles-tokenized\AA\wiki_06.bz2
C:\Users\Wei Khan

In [26]:
sample_file = bz2_files[0]

with bz2.open(sample_file, "rt", encoding="utf-8") as f:
    first_line = f.readline()

article = json.loads(first_line)

print("Available fields:", article.keys())
print("Article ID:", article.get("id"))
print("Title:", article.get("title"))
print("URL:", article.get("url"))
print("Text type:", type(article.get("text")))

Available fields: dict_keys(['id', 'url', 'title', 'text', 'offsets', 'text_with_links', 'offsets_with_links'])
Article ID: 12
Title: Anarchism
URL: https://en.wikipedia.org/wiki?curid=12
Text type: <class 'list'>


## 3. Cleaning and Chunking

In [27]:
def clean_article_text(article):
    title = article.get("title", "").strip()
    raw_text = article.get("text", [])

    cleaned_sentences = []

    for item in raw_text:
        if not isinstance(item, str):
            continue

        sentence = " ".join(item.split())

        if not sentence:
            continue

        if sentence.lower() == title.lower():
            continue

        cleaned_sentences.append(sentence)

    return cleaned_sentences

In [28]:
# Split cleaned text into 100-word chunks with 20-word overlap while preserving article metadata.
def create_chunks(
    sentences,
    article_id,
    title,
    url,
    max_words=100,
    overlap_words=20
):
    chunks = []
    current_words = []

    for sentence in sentences:
        words = sentence.split()

        while words:
            remaining_space = max_words - len(current_words)

            if len(words) <= remaining_space:
                current_words.extend(words)
                words = []
            else:
                current_words.extend(words[:remaining_space])

                chunks.append({
                    "passage_id": f"{article_id}_{len(chunks)}",
                    "article_id": article_id,
                    "title": title,
                    "url": url,
                    "text": " ".join(current_words),
                })

                overlap = current_words[-overlap_words:] if overlap_words > 0 else []
                current_words = overlap
                words = words[remaining_space:]

    if current_words:
        chunks.append({
            "passage_id": f"{article_id}_{len(chunks)}",
            "article_id": article_id,
            "title": title,
            "url": url,
            "text": " ".join(current_words),
        })

    return chunks

In [29]:
# Clean a Wikipedia article and convert it into text chunks for retrieval.
def article_to_chunks(article):
    cleaned_sentences = clean_article_text(article)

    if not cleaned_sentences:
        return []

    return create_chunks(
        sentences=cleaned_sentences,
        article_id=article.get("id"),
        title=article.get("title", ""),
        url=article.get("url", ""),
        max_words=100,
        overlap_words=20,
    )

## 4. Load or Build the 1,000-Article Prototype Corpus

In [30]:
if MINI_CORPUS_PATH.exists():
    print("Saved mini corpus found. Loading it...")

    mini_corpus = []
    with open(MINI_CORPUS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            mini_corpus.append(json.loads(line))

else:
    print("Saved mini corpus not found. Building from the first 1,000 Wikipedia articles...")

    MAX_ARTICLES = 1000
    mini_corpus = []
    processed_articles = 0
    bad_records = 0

    for file_path in bz2_files:
        with bz2.open(file_path, "rt", encoding="utf-8", errors="replace") as f:
            for line in f:
                try:
                    article = json.loads(line)
                    mini_corpus.extend(article_to_chunks(article))
                    processed_articles += 1
                except json.JSONDecodeError:
                    bad_records += 1
                    continue

                if processed_articles >= MAX_ARTICLES:
                    break

        if processed_articles >= MAX_ARTICLES:
            break

    with open(MINI_CORPUS_PATH, "w", encoding="utf-8") as f:
        for chunk in mini_corpus:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print("Articles processed:", processed_articles)
    print("Bad records skipped:", bad_records)
    print("Saved:", MINI_CORPUS_PATH)

print("Prototype passages:", len(mini_corpus))

Saved mini corpus found. Loading it...
Prototype passages: 35822


In [31]:
passage_ids = [chunk["passage_id"] for chunk in mini_corpus]
word_counts = [len(chunk["text"].split()) for chunk in mini_corpus]

print("Unique passage IDs:", len(set(passage_ids)))
print("Maximum words:", max(word_counts))
assert len(passage_ids) == len(set(passage_ids))
assert max(word_counts) <= 100
print("PASS: prototype corpus validation successful.")

Unique passage IDs: 35822
Maximum words: 100
PASS: prototype corpus validation successful.


## 5. BM25 Sparse Retrieval

In [32]:
def bm25_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


bm25_documents = [
    chunk["title"] + " " + chunk["text"]
    for chunk in mini_corpus
]

tokenized_corpus = [
    bm25_tokenize(document)
    for document in bm25_documents
]

bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index created:", len(tokenized_corpus), "passages")

BM25 index created: 35822 passages


In [33]:
def bm25_search(query, top_k=10):
    tokenized_query = bm25_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = mini_corpus[idx]

        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "bm25_score": float(scores[idx]),
        })

    return results

## 6. DPR + FAISS Dense Retrieval

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

ctx_model_name = "facebook/dpr-ctx_encoder-multiset-base"
question_model_name = "facebook/dpr-question_encoder-multiset-base"

ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(ctx_model_name)
ctx_encoder = DPRContextEncoder.from_pretrained(ctx_model_name).to(device)

question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(question_model_name)
question_encoder = DPRQuestionEncoder.from_pretrained(question_model_name).to(device)

ctx_encoder.eval()
question_encoder.eval()

print("DPR models loaded.")

Device: cpu


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12312.29it/s]
[transformers] DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-multiset-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 32773.20it/s]
[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored 

DPR models loaded.


In [35]:
def encode_passages(documents, tokenizer, model, batch_size=32):
    embeddings = []

    for start in range(0, len(documents), batch_size):
        batch_texts = documents[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

            batch_embeddings = (
                outputs.pooler_output
                .cpu()
                .numpy()
                .astype("float32")
            )

        embeddings.append(batch_embeddings)

        print(
            f"Encoded {min(start + batch_size, len(documents))} / "
            f"{len(documents)} passages"
        )

    return np.vstack(embeddings)

In [36]:
dpr_documents = [
    chunk["title"] + " " + chunk["text"]
    for chunk in mini_corpus
]

OLD_EMBEDDINGS_PATH = OUTPUT_DIR / "old_dpr_passage_embeddings.npy"

if OLD_EMBEDDINGS_PATH.exists():
    passage_embeddings = np.load(OLD_EMBEDDINGS_PATH)

    if (
        passage_embeddings.ndim != 2
        or passage_embeddings.shape[0] != len(mini_corpus)
        or passage_embeddings.shape[1] != 768
    ):
        print("Existing embeddings do not match the corpus. Regenerating...")
        passage_embeddings = encode_passages(
            dpr_documents, ctx_tokenizer, ctx_encoder, batch_size=64
        )
        np.save(OLD_EMBEDDINGS_PATH, passage_embeddings)
    else:
        print("Loaded saved prototype embeddings:", passage_embeddings.shape)
else:
    passage_embeddings = encode_passages(
        dpr_documents, ctx_tokenizer, ctx_encoder, batch_size=64
    )
    np.save(OLD_EMBEDDINGS_PATH, passage_embeddings)
    print("Saved prototype embeddings:", OLD_EMBEDDINGS_PATH)

Loaded saved prototype embeddings: (35822, 768)


In [37]:
faiss_index = faiss.IndexFlatIP(passage_embeddings.shape[1])
faiss_index.add(passage_embeddings)

assert faiss_index.ntotal == len(mini_corpus)
print("FAISS passages indexed:", faiss_index.ntotal)

FAISS passages indexed: 35822


In [38]:
def dpr_search(query, top_k=10):
    inputs = question_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        query_embedding = (
            question_encoder(**inputs)
            .pooler_output
            .cpu()
            .numpy()
            .astype("float32")
        )

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        chunk = mini_corpus[idx]

        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "dpr_score": float(score),
        })

    return results

## 7. Hybrid Retrieval and Optional Reranker

In [39]:
def hybrid_search(query, top_k_each=10):
    bm25_results = bm25_search(query, top_k=top_k_each)
    dpr_results = dpr_search(query, top_k=top_k_each)

    combined = {}

    for result in bm25_results:
        pid = result["passage_id"]
        combined[pid] = {
            "passage_id": pid,
            "article_id": result["article_id"],
            "title": result["title"],
            "text": result["text"],
            "bm25_score": result["bm25_score"],
            "dpr_score": None,
            "source": ["BM25"],
        }

    for result in dpr_results:
        pid = result["passage_id"]

        if pid in combined:
            combined[pid]["dpr_score"] = result["dpr_score"]
            combined[pid]["source"].append("DPR")
        else:
            combined[pid] = {
                "passage_id": pid,
                "article_id": result["article_id"],
                "title": result["title"],
                "text": result["text"],
                "bm25_score": None,
                "dpr_score": result["dpr_score"],
                "source": ["DPR"],
            }

    return list(combined.values())

In [40]:
reranker_model_name = "BAAI/bge-reranker-large"

reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_model_name)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    reranker_model_name
).to(device)

reranker_model.eval()


def rerank_candidates(query, candidates, top_k=10, batch_size=4):
    if not candidates:
        return []

    passages = [
        f"{candidate['title']}. {candidate['text']}"
        for candidate in candidates
    ]

    all_scores = []

    for start in range(0, len(passages), batch_size):
        batch_passages = passages[start:start + batch_size]
        batch_queries = [query] * len(batch_passages)

        inputs = reranker_tokenizer(
            batch_queries,
            batch_passages,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = reranker_model(**inputs)
            scores = outputs.logits.view(-1).float().cpu().tolist()

        all_scores.extend(scores)

    reranked = []

    for candidate, score in zip(candidates, all_scores):
        result = candidate.copy()
        result["reranker_score"] = float(score)
        reranked.append(result)

    reranked.sort(key=lambda x: x["reranker_score"], reverse=True)
    return reranked[:top_k]

Loading weights: 100%|██████████| 393/393 [00:01<00:00, 210.44it/s]


## 8. Standard RAG Hybrid Fusion (RRF)

In [41]:
def standard_hybrid_search(
    query,
    top_k_each=10,
    final_top_k=10,
    rrf_k=60
):
    bm25_results = bm25_search(query, top_k=top_k_each)
    dpr_results = dpr_search(query, top_k=top_k_each)

    combined = {}

    for rank, result in enumerate(bm25_results, start=1):
        pid = result["passage_id"]

        if pid not in combined:
            combined[pid] = {
                "passage_id": pid,
                "article_id": result["article_id"],
                "title": result["title"],
                "text": result["text"],
                "bm25_rank": None,
                "dpr_rank": None,
                "bm25_score": None,
                "dpr_score": None,
                "rrf_score": 0.0,
                "source": [],
            }

        combined[pid]["bm25_rank"] = rank
        combined[pid]["bm25_score"] = result["bm25_score"]
        combined[pid]["source"].append("BM25")
        combined[pid]["rrf_score"] += 1 / (rrf_k + rank)

    for rank, result in enumerate(dpr_results, start=1):
        pid = result["passage_id"]

        if pid not in combined:
            combined[pid] = {
                "passage_id": pid,
                "article_id": result["article_id"],
                "title": result["title"],
                "text": result["text"],
                "bm25_rank": None,
                "dpr_rank": None,
                "bm25_score": None,
                "dpr_score": None,
                "rrf_score": 0.0,
                "source": [],
            }

        combined[pid]["dpr_rank"] = rank
        combined[pid]["dpr_score"] = result["dpr_score"]

        if "DPR" not in combined[pid]["source"]:
            combined[pid]["source"].append("DPR")

        combined[pid]["rrf_score"] += 1 / (rrf_k + rank)

    ranked_results = sorted(
        combined.values(),
        key=lambda x: x["rrf_score"],
        reverse=True,
    )

    return ranked_results[:final_top_k]

## 9. DeepSeek Generator

The original notebook contained a plaintext API key. It is intentionally **not copied here**.
Enter the API key securely when prompted.

In [42]:
# API Key: sk-0ea11e890a374b478bccdc872bbdf1f5
if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass(
        "Enter your DeepSeek API key: "
    )

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

DEEPSEEK_MODEL = "deepseek-v4-flash"
print("DeepSeek client ready.")

DeepSeek client ready.


In [43]:
def build_standard_context(retrieved_results, max_passages=10):
    context_blocks = []

    for i, result in enumerate(retrieved_results[:max_passages], start=1):
        context_blocks.append(
            f"[Passage {i}]\n"
            f"Title: {result['title']}\n"
            f"Text: {result['text']}"
        )

    return "\n\n".join(context_blocks)


def generate_standard_answer(question, context, model=DEEPSEEK_MODEL):
    system_prompt = """
You are an open-domain question answering system.

Answer the question using only the retrieved Wikipedia context provided.

Rules:
1. Use the retrieved context as the evidence for your answer.
2. Do not use unsupported outside information.
3. Return only the short factual answer.
4. Do not provide explanations unless the question requires them.
5. If the retrieved context does not contain enough evidence to answer the question, return:
INSUFFICIENT_EVIDENCE
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved Wikipedia Context:
{context}

Answer:
""".strip()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=100,
        temperature=0.0,
        extra_body={"thinking": {"type": "disabled"}},
    )

    return response.choices[0].message.content.strip()

In [44]:
def standard_rag(
    question,
    top_k_each=10,
    final_top_k=10,
    model=DEEPSEEK_MODEL
):
    retrieved_results = standard_hybrid_search(
        query=question,
        top_k_each=top_k_each,
        final_top_k=final_top_k,
    )

    context = build_standard_context(
        retrieved_results,
        max_passages=final_top_k,
    )

    generated_answer = generate_standard_answer(
        question=question,
        context=context,
        model=model,
    )

    return {
        "question": question,
        "generated_answer": generated_answer,
        "retrieved_passages": retrieved_results,
        "context": context,
    }

## 10. Prototype Smoke Test

In [ ]:
test_question = (
    "What political philosophy rejects involuntary, "
    "coercive forms of hierarchy?"
    #"Which genus contains more species, Ortegocactus or Eschscholzia?"
)

result = standard_rag(test_question)
print("Question:", result["question"])
print("Answer:", result["generated_answer"])

Question: Which genus contains more species, Ortegocactus or Eschscholzia?
Answer: INSUFFICIENT_EVIDENCE


In [46]:
print("\nRETRIEVED PASSAGES")

for rank, passage in enumerate(
    result["retrieved_passages"],
    start=1
):
    print(
        rank,
        "|",
        passage["passage_id"],
        "|",
        passage["title"],
        "| RRF:",
        round(passage["rrf_score"], 6)
    )


RETRIEVED PASSAGES
1 | 12_0 | Anarchism | RRF: 0.032787
2 | 12_33 | Anarchism | RRF: 0.031514
3 | 1023_86 | Anarcho-capitalism | RRF: 0.016129
4 | 1078_31 | Antisemitism | RRF: 0.015873
5 | 12_6 | Anarchism | RRF: 0.015873
6 | 339_38 | Ayn Rand | RRF: 0.015625
7 | 1023_5 | Anarcho-capitalism | RRF: 0.015625
8 | 1023_12 | Anarcho-capitalism | RRF: 0.015385
9 | 894_6 | Agnosticism | RRF: 0.015152
10 | 1023_35 | Anarcho-capitalism | RRF: 0.015152
